In [41]:
import requests
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import re

donnees = []
scraper = cloudscraper.create_scraper()

for j in range(1,30):
    if j==1:  
        page = requests.get("https://www.etreproprio.com/annonces/thf.ld75#list")
    else:
        page = requests.get("https://www.etreproprio.com/annonces/thf.ld75.odd.g"+str(j)+"#list")
        
    soup = BeautifulSoup(page.content, "html.parser")
    temp = soup.find("div", class_='ep-search-list-wrapper').find_all("a")
    
    liens = []
    for i in range(len(temp)):
        if "https://www.etreproprio.com/immobilier-" in temp[i]['href']:
            liens.append(temp[i]['href'])
    
    for lien in liens:
        try:
            page_annonce = scraper.get(lien)
            soup_ = BeautifulSoup(page_annonce.content, 'lxml')

            try:
                # "Vente Appartement 75 m² Paris-14e"
                titre = soup_.find("title").text.strip()
            except:
                titre = None

            try:
                # "Appartement 75 m² à Paris-14e"
                description_h1 = soup_.find("h1", class_='annonce-immobilier').text.strip()
            except:
                description_h1 = None

            try:
                # Extrait la ville depuis le titre : "Vente Appartement 75 m² Paris-14e" → "Paris-14e"
                ville = soup_.find("title").text.strip().split("²")[-1].strip()
            except:
                ville = None

            try:
                prix = soup_.find("div", class_='ep-price').text.strip()
                prix = int(prix.replace(" ", "").replace("\xa0", "").replace("€", ""))
            except:
                prix = None

            try:
                surface = soup_.find("div", class_='ep-area').text.strip()
                surface = int(surface.replace("\xa0", "").replace("m²", "").strip())
            except:
                surface = None

            try:
                type_bien = titre.split(" ")[1]  # le 2ème mot après "Vente"
            except:
                type_bien = None

            try:
                desc_texte = soup_.find("div", class_='ep-desc-truncated').text.strip()
            except:
                desc_texte = None
            
            try:
                etage = None
                if desc_texte:
                # Cas 1 — chiffres
                    etage_match = re.search(r'(\d+)\s*(?:[eèéê][mr]?e?|°)?\s*étage', desc_texte, re.IGNORECASE)
                    if etage_match:
                        etage = int(etage_match.group(1))
                # Cas 2 — lettres
                    else:
                        etages_lettres = {
                            "premier": 1, "première": 1,
                            "deuxième": 2, "deuxieme": 2,
                            "troisième": 3, "troisieme": 3,
                            "quatrième": 4, "quatrieme": 4,
                            "cinquième": 5, "cinquieme": 5,
                            "sixième": 6, "sixieme": 6,
                            "septième": 7, "septieme": 7,
                            "huitième": 8, "huitieme": 8,
                            "neuvième": 9, "neuvieme": 9,
                            "dixième": 10, "dixieme": 10,
                        }
                        for mot, num in etages_lettres.items():
                            if mot in desc_texte.lower():
                                etage = num
                                break
                # Cas 3 — rez-de-chaussée
                    if etage is None:
                        rdc_patterns = ["rez-de-chaussée", "rez de chaussée","rez-de-chaussee", "rdc", "r.d.c"]
                        if any(p in desc_texte.lower() for p in rdc_patterns):
                            etage = 0
            except:
                etage = None

            try:
                dpe = soup_.find("div", class_='dpe-diagram').find("div", class_='selected').text.strip()
            except:
                dpe = None

            try:
                pieces = soup_.find("div", class_='ep-room').text.strip()
                pieces = int(pieces.replace("pièces", "").replace("pièce", "").strip())
            except:
                pieces = None

            print(f"Titre: {titre} | Ville: {ville} | Prix: {prix} | Surface: {surface} | Type: {type_bien} | Etage: {etage} | Pièces: {pieces} | DPE: {dpe}")

            donnees.append({
                "titre": titre,
                "description": description_h1,
                "ville": ville,
                "type_bien": type_bien,
                "etage": etage,
                "prix": prix,
                "surface": surface,
                "pieces": pieces,
                "DPE": dpe,
                "full_description": desc_texte,
                "lien": lien
            })

        except Exception as e:
            print(f"Erreur sur {lien} : {e}")
            continue

# Création du DataFrame
df = pd.DataFrame(donnees)
df["prix"] = pd.to_numeric(df["prix"], errors="coerce")
df["surface"] = pd.to_numeric(df["surface"], errors="coerce")
df["pieces"] = pd.to_numeric(df["pieces"], errors="coerce")
df["etage"] = pd.to_numeric(df["etage"], errors="coerce")

print(df)
df.to_csv("annonces_paris.csv", index=False, encoding="utf-8-sig")

Titre: Vente Appartement 46 m² Paris-13e | Ville: Paris-13e | Prix: 445000 | Surface: 46 | Type: Appartement | Etage: None | Pièces: 3 | DPE: E
Titre: Vente Maison 130 m² Paris-14e | Ville: Paris-14e | Prix: 1298000 | Surface: 130 | Type: Maison | Etage: None | Pièces: 5 | DPE: B
Titre: Vente Maison 170 m² Paris-20e | Ville: Paris-20e | Prix: 925000 | Surface: None | Type: Maison | Etage: 1 | Pièces: 6 | DPE: D
Titre: Vente Appartement 9 m² Paris-5e | Ville: Paris-5e | Prix: 133000 | Surface: 9 | Type: Appartement | Etage: None | Pièces: 1 | DPE: F
Titre: Vente Appartement 64 m² Paris-12e | Ville: Paris-12e | Prix: 953900 | Surface: 64 | Type: Appartement | Etage: None | Pièces: 3 | DPE: None
Titre: Vente Appartement 39 m² Paris-17e | Ville: Paris-17e | Prix: 349500 | Surface: 39 | Type: Appartement | Etage: 1 | Pièces: 2 | DPE: D
Titre: Vente Appartement 64 m² Paris-17e | Ville: Paris-17e | Prix: 970000 | Surface: 64 | Type: Appartement | Etage: 2 | Pièces: 3 | DPE: D
Titre: Vente App